# Section 3 - Anomaly Detection for Cyber Threats

This notebook cleans an unmodified CSE-CIC-IDS2018 daily flow CSV and compares a benign-only Isolation Forest with a dense Autoencoder. Labels are reserved for validation threshold selection and final evaluation.

## 1. Locate or clone the project

Locally, the cell finds the current checkout. In a fresh Colab runtime, it automatically clones the GitHub repository into `/content/ml_assignment`. Push local changes before starting Colab so the remote clone contains them.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

GITHUB_OWNER = "Saumyakeshi"
GITHUB_REPOSITORY = "ml_assignment"
REPOSITORY_URL = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPOSITORY}.git"
COLAB_PROJECT_DIR = Path("/content/ml_assignment")

def is_project(path: Path) -> bool:
    return (path / "pyproject.toml").is_file() and (path / "src" / "comp70049").is_dir()

def find_local_project() -> Path | None:
    current = Path.cwd().resolve()
    return next((candidate for candidate in [current, *current.parents] if is_project(candidate)), None)

running_on_colab = Path("/content").is_dir()
if running_on_colab:
    if not is_project(COLAB_PROJECT_DIR):
        if COLAB_PROJECT_DIR.exists():
            raise FileExistsError(
                f"{COLAB_PROJECT_DIR} exists but is incomplete. Restart the runtime and rerun."
            )
        subprocess.run(["git", "clone", REPOSITORY_URL, str(COLAB_PROJECT_DIR)], check=True)
    PROJECT_DIR = COLAB_PROJECT_DIR
else:
    PROJECT_DIR = find_local_project()
    if PROJECT_DIR is None:
        raise FileNotFoundError("Open the notebook from inside the repository.")

os.chdir(PROJECT_DIR)
SOURCE_DIR = PROJECT_DIR / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
print("Environment:", "Google Colab" if running_on_colab else "Local")
print("Project:", PROJECT_DIR)

## 2. Install dependencies

In [ ]:
%pip install -q -e ".[deep]"

In [ ]:
import copy
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from comp70049.anomaly_detection.autoencoder import train_and_evaluate_autoencoder
from comp70049.anomaly_detection.classic import evaluate_isolation_forest, train_isolation_forest
from comp70049.anomaly_detection.data import create_anomaly_partitions, load_cic_ids2018, profile_partition
from comp70049.anomaly_detection.preprocessing import build_feature_pipeline, fit_transform_features, transformed_feature_names

config = json.loads(Path("configs/section_03.json").read_text(encoding="utf-8"))
config

## 3. Download the official unclean daily CSV

The downloader verifies the 103 MiB file's size and SHA-256 checksum. Raw data remains ignored by Git.

In [ ]:
subprocess.run([sys.executable, "scripts/download_cic_ids2018.py"], check=True)

## 4. Audit and clean the raw data

The cleaning step removes exact duplicates and an embedded header, replaces infinities, coerces malformed numeric cells, reports invalid timestamps, and retains statistical outliers.

In [ ]:
cleaned = load_cic_ids2018(
    Path(config["data_file"]),
    maximum_missing_fraction=float(config["maximum_missing_fraction"]),
)
quality = {**cleaned.raw_profile, **cleaned.cleaning_report}
display(pd.Series(quality, name="value").to_frame())
display(pd.Series(cleaned.cleaning_report["attack_label_counts"], name="records").to_frame())

In [ ]:
issues = pd.Series({
    "Duplicate rows": cleaned.cleaning_report["duplicate_records_removed"],
    "Embedded headers": cleaned.cleaning_report["embedded_header_rows_removed"],
    "Infinite values": cleaned.cleaning_report["infinite_values_replaced_with_missing"],
    "Non-numeric values": cleaned.cleaning_report["non_numeric_values_coerced_to_missing"],
    "Invalid timestamps": cleaned.cleaning_report["invalid_timestamps"],
    "Remaining missing": cleaned.cleaning_report["remaining_missing_values"],
})
issues.plot.bar(color="#F2CF5B", title="Data-quality issues handled", ylabel="Values or rows", rot=25)
plt.grid(axis="y", alpha=0.25)
plt.show()

## 5. Create the anomaly-detection partitions

Training contains only benign records. Validation and test contain both benign and labelled infiltration records.

In [ ]:
partitions = create_anomaly_partitions(
    cleaned.frame,
    seed=int(config["seed"]),
    max_benign_records=int(config["max_benign_records"]),
    max_anomaly_records=int(config["max_anomaly_records"]),
    benign_train_fraction=float(config["benign_train_fraction"]),
    benign_validation_fraction=float(config["benign_validation_fraction"]),
    anomaly_validation_fraction=float(config["anomaly_validation_fraction"]),
)
profiles = {name: profile_partition(frame) for name, frame in {
    "train": partitions.train,
    "validation": partitions.validation,
    "test": partitions.test,
}.items()}
display(pd.DataFrame(profiles).T[["records", "benign", "anomaly", "missing_values"]])

## 6. Fit preprocessing on benign training data only

In [ ]:
pipeline = build_feature_pipeline()
train_features, validation_features, test_features = fit_transform_features(
    pipeline,
    cleaned.feature_columns,
    partitions.train,
    partitions.validation,
    partitions.test,
)
feature_names = transformed_feature_names(pipeline, cleaned.feature_columns)
validation_labels = partitions.validation["label"].to_numpy(dtype=np.int64)
test_labels = partitions.test["label"].to_numpy(dtype=np.int64)
results_dir = Path(config["results_dir"])
models_dir = Path(config["models_dir"])
print("Matrices:", train_features.shape, validation_features.shape, test_features.shape)
print("Transformed features:", len(feature_names))

## 7. Isolation Forest

In [ ]:
forest = train_isolation_forest(
    train_features,
    config=config["isolation_forest"],
    seed=int(config["seed"]),
)
forest_metrics = evaluate_isolation_forest(
    forest, pipeline, validation_features, validation_labels, test_features, test_labels,
    feature_names=feature_names, model_dir=models_dir, results_dir=results_dir,
)
display(pd.Series(forest_metrics).drop("confusion_matrix").to_frame("value"))
display(Image(filename=str(results_dir / "figures/isolation-forest-confusion-matrix.png")))
display(Image(filename=str(results_dir / "figures/isolation-forest-precision-recall-curve.png")))

## 8. Autoencoder

Keep `AUTOENCODER_EPOCHS = None` for the configured full experiment with early stopping. Change it to `1` only for a quick pipeline check.

In [ ]:
AUTOENCODER_EPOCHS = None
autoencoder_config = copy.deepcopy(config["autoencoder"])
if AUTOENCODER_EPOCHS is not None:
    autoencoder_config["epochs"] = AUTOENCODER_EPOCHS

autoencoder_metrics = train_and_evaluate_autoencoder(
    train_features, validation_features, validation_labels, test_features, test_labels,
    feature_names=feature_names, config=autoencoder_config, seed=int(config["seed"]),
    model_dir=models_dir, results_dir=results_dir,
)
display(pd.Series(autoencoder_metrics).drop(["confusion_matrix", "training_history"]).to_frame("value"))
display(Image(filename=str(results_dir / "figures/autoencoder-confusion-matrix.png")))
display(Image(filename=str(results_dir / "figures/autoencoder-precision-recall-curve.png")))
display(Image(filename=str(results_dir / "figures/autoencoder-training-history.png")))

## 9. Compare the models

A high TPR is not useful when FPR is also high. Interpret the ranking metrics and threshold trade-off together.

In [ ]:
metric_names = ["true_positive_rate", "false_positive_rate", "precision", "f1", "average_precision", "roc_auc"]
comparison = pd.DataFrame([
    {"model": "Isolation Forest", **{name: forest_metrics[name] for name in metric_names}},
    {"model": f"Autoencoder ({autoencoder_metrics['epochs_completed']} epoch(s))", **{name: autoencoder_metrics[name] for name in metric_names}},
])
comparison.to_csv(results_dir / "model-comparison.csv", index=False)
display(comparison.style.format(precision=4))

## 10. Save Colab results

Colab files are remote. Download this archive before the runtime resets, or commit the generated report artifacts separately.

In [ ]:
import shutil

if running_on_colab:
    archive = shutil.make_archive("/content/section_03_results", "zip", root_dir=results_dir)
    print("Download from the Colab Files panel:", archive)
else:
    print("Results are already local at:", results_dir.resolve())